# Pass Clustering & Team Playing Style Visualisation  

**Competition Focus:** La Liga - 2015/16 Season  
**Dataset:** StatsBomb Open Data (free event data repository)  
**Purpose:** Build pass clustering visualisations that reveal team playing styles, using event data to identify recurring passing patterns and present them in an intuitive, insight-driven format.  
**Methods:** Feature engineering, unsupervised learning (K-Means clustering), distributional analysis, z-score normalisation, and visualisation using representative pass trajectories.  
**Author:** [Victoria Friss de Kereki](https://www.linkedin.com/in/victoria-friss-de-kereki/)  
**Medium Articles:**  _TBC_

**Notebook first written:** `14/04/2026`  
**Last updated:** `15/04/2026`  

> This notebook presents a step-by-step approach to building pass clustering visualisations using event data from the 2015/16 La Liga season. Each pass is represented by its start and end coordinates and grouped into clusters using K-Means, allowing recurring “pass types” to emerge from the data.  
>
> To make these patterns interpretable at team level, cluster usage is aggregated and compared to league averages using z-scores, highlighting which passing behaviours are over- or under-represented.  
>
> The approach is applied across multiple contexts, including all passes, own half build-up, and defensive third build-up, enabling a structured view of how teams construct possession in different phases of play.  
>
> The objective is not to evaluate performance, but to demonstrate how clustering and visualisation techniques can be combined to better understand and communicate patterns in football data.  

  
---------------------

## 1. Packages and Configuration

In [ ]:
# Standard library
import math
import warnings
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
from mplsoccer import Pitch

# Football data
from statsbombpy import sb

# Machine learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Utilities
from tqdm import tqdm

warnings.filterwarnings("ignore", message="credentials were not supplied")

## 2. Load Competition, Match & Event Data

In [ ]:
# Download Matches

leagues = [(11, 27)]  # La Liga 2015/16

matches_df = pd.concat([
    sb.matches(competition_id=comp, season_id=season)
    for comp, season in leagues
], ignore_index=True)

len(matches_df)

In [ ]:
# Extract Pass Events

CACHE_DIR = Path("statsbomb_cache/events_15_16")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_events(match_id):
    cache_file = CACHE_DIR / f"{match_id}.parquet"
    if cache_file.exists():
        return pd.read_parquet(cache_file)

    events = sb.events(match_id=match_id)
    events.to_parquet(cache_file)
    return events

all_passes = []

for match_id in tqdm(matches_df["match_id"], desc="Extracting passes"):
    events = load_events(match_id)
    passes = events[events["type"] == "Pass"].copy()
    passes["match_id"] = match_id
    all_passes.append(passes)

passes_df = pd.concat(all_passes, ignore_index=True)

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

passes_df.head()

## 3. Data Cleaning & Feature Engineering

In [ ]:
passes = passes_df.dropna(axis=1, how="all").copy()

def extract_coords(col):
    x = pd.to_numeric(col.astype(str).str.strip("[]").str.split().str[0], errors="coerce")
    y = pd.to_numeric(col.astype(str).str.strip("[]").str.split().str[1], errors="coerce")
    return x, y

passes["start_x"], passes["start_y"] = extract_coords(passes["location"])
passes["end_x"], passes["end_y"] = extract_coords(passes["pass_end_location"])

# Remove unwanted passes
passes = passes[passes["pass_outcome"] != "Injury Clearance"].copy()

# Fix pitch bounds
PITCH_LENGTH, PITCH_WIDTH = 120, 80

for col, max_val in [("start_x", 120), ("end_x", 120), ("start_y", 80), ("end_y", 80)]:
    passes[col] = passes[col].clip(lower=0, upper=max_val)

## 4. Clustering Framework (Reusable)

In [ ]:
FEATURES = ["start_x", "start_y", "end_x", "end_y"]

def run_clustering(df, n_clusters=50, seed=14):
    scaler = StandardScaler()
    X = scaler.fit_transform(df[FEATURES])

    model = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    df = df.copy()
    df["cluster"] = model.fit_predict(X)

    return df, scaler

## 5. Cluster Analysis Utilities

In [ ]:
def compute_zscores(df):
    team_cluster = (
        df.groupby(["team", "cluster"])
        .size()
        .reset_index(name="count")
    )

    team_total = df.groupby("team").size().reset_index(name="total")

    dist = team_cluster.merge(team_total, on="team")
    dist["prop"] = dist["count"] / dist["total"]

    league_avg = dist.groupby("cluster")["prop"].mean()
    league_std = dist.groupby("cluster")["prop"].std()

    dist["z"] = (
        (dist["prop"] - dist["cluster"].map(league_avg)) /
        (dist["cluster"].map(league_std) + 1e-6)
    )

    return dist

## 6. Representative Passes & Descriptions

In [ ]:
def representative_passes(df_cluster, scaler, n=30):
    X = scaler.transform(df_cluster[FEATURES])
    centroid = X.mean(axis=0)

    dist = np.linalg.norm(X - centroid, axis=1)
    df_cluster = df_cluster.copy()
    df_cluster["dist"] = dist

    return df_cluster.sort_values("dist").head(n)

## 7. Visualisation Functions

In [ ]:
pitch = Pitch(pitch_type="statsbomb", line_color="black")


def plot_team(ax, df, dist, scaler, team, display_name=None, top_n=5, show_descriptions=False):
    team_df = df[df["team"] == team]

    # --- handle z column naming ---
    z_col = "z_score" if "z_score" in dist.columns else "z"

    top_clusters = (
        dist[dist["team"] == team]
        .sort_values(z_col, ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    cmap = plt.colormaps["tab10"]

    # --- stable cluster-color mapping ---
    cluster_info = []
    for i, row in enumerate(top_clusters.itertuples()):
        cluster_info.append({
            "cluster": row.cluster,
            "color": cmap(i % 10),
            "idx": i + 1
        })

    # --- plot in reverse so #1 is on top ---
    for info in reversed(cluster_info):
        cluster_df = team_df[team_df["cluster"] == info["cluster"]]
        reps = representative_passes(cluster_df, scaler)

        for _, p in reps.iterrows():
            ax.arrow(
                p["start_x"], p["start_y"],
                p["end_x"] - p["start_x"],
                p["end_y"] - p["start_y"],
                head_width=1.2,
                head_length=2,
                linewidth=1.3,
                color=info["color"],
                alpha=0.4
            )

    # --- legend (original order) ---
    descriptions = [(c["idx"], c["color"]) for c in cluster_info]

    # --- title (with optional position) ---
    title = display_name if display_name is not None else team
    ax.set_title(title, pad=6, fontweight="bold")

    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

    y = -0.02

    if show_descriptions:
        for i, (idx, color) in enumerate(descriptions):
            ax.text(
                0.5,
                y - i * 0.06,
                f"➜ {idx}",
                transform=ax.transAxes,
                fontsize=9,
                color=color,
                ha="center",
                va="top"
            )
    else:
        x_positions = np.linspace(0.2, 0.8, len(descriptions))

        for x, (idx, color) in zip(x_positions, descriptions):
            ax.text(
                x,
                y,
                f"➜ {idx}",
                transform=ax.transAxes,
                fontsize=11,
                color=color,
                ha="center",
                va="top"
            )


def plot_grid(df, dist, scaler, title, n_cols=4, show_descriptions=False):

    TEAM_ORDER = [
        "Barcelona",
        "Real Madrid",
        "Atlético Madrid",
        "Villarreal",
        "Athletic Club",
        "Celta Vigo",
        "Sevilla",
        "Málaga",
        "Real Sociedad",
        "Real Betis",
        "Las Palmas",
        "Valencia",
        "Espanyol",
        "Eibar",
        "RC Deportivo La Coruña",
        "Granada",
        "Sporting Gijón",
        "Rayo Vallecano",
        "Getafe",
        "Levante UD"
    ]

    # --- enforce league order ---
    teams = [t for t in TEAM_ORDER if t in df["team"].unique()]

    n_rows = math.ceil(len(teams) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 4 * n_rows),
        gridspec_kw={'hspace': 0.08, 'wspace': 0.05}
    )

    axes = axes.flatten()

    fig.suptitle(
        title,
        fontsize=16,
        fontweight="bold",
        y=0.91
    )

    for i, team in enumerate(teams):
        pitch.draw(ax=axes[i])

        plot_team(
            axes[i],
            df,
            dist,
            scaler,
            team,
            display_name=f"{i+1}. {team}",  # 👈 position added here
            show_descriptions=show_descriptions
        )

    # remove empty plots
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.show()

## 8. Applications

In [ ]:
# ALL PASSES

df_all, scaler = run_clustering(passes)
dist_all = compute_zscores(df_all)

plot_grid(
    df_all,
    dist_all,
    scaler,
    "Team Passing Style Clusters (All Passes) | La Liga 2015/16 | Top 5 Clusters (Z-Score vs League)"
)

In [ ]:
# OWN HALF

own_half = passes[passes["start_x"] <= 60].copy()

df_own, scaler = run_clustering(own_half)
dist_own = compute_zscores(df_own)

plot_grid(
    df_own,
    dist_own,
    scaler,
    "Team Passing Style Clusters (Own Half) | La Liga 2015/16 | Top 5 Clusters (Z-Score vs League)"
)

In [ ]:
# DEFENSIVE THIRD

def_third = passes[passes["start_x"] <= 40].copy()

df_def, scaler = run_clustering(def_third)
dist_def = compute_zscores(df_def)

plot_grid(
    df_def,
    dist_def,
    scaler,
    "Team Passing Style Clusters (Defensive Third) | La Liga 2015/16 | Top 5 Clusters (Z-Score vs League)"
)